# INCA Training — Validación externa

Notebook de entrenamiento para el dataset INCA.


In [ ]:
# ============================================================
# SETUP — run this cell once after every runtime restart
# Do NOT run training here. Select one experiment cell below.
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

!git clone https://github.com/sebastianquispearias/tesis-seg.git
%cd tesis-seg
!pip install -q -r requirements.txt

# --- Environment fingerprint (for reproducibility debugging) ---
import torch, sys
print("Python    :", sys.version)
print("torch     :", torch.__version__)
print("CUDA      :", torch.version.cuda)
!git log --oneline -1
!pip show albumentations | grep Version
!nvidia-smi | grep -E "NVIDIA|Driver Version|CUDA Version"
# --------------------------------------------------------------

import sys
sys.path.append("/content/tesis-seg")

from src.defaults import get_default_config, summarize_config
from src.augmentations import (
    get_supervised_train_augmentation,
    get_weak_augmentation,
    get_strong_augmentation,
)
from src.datasets import (
    build_supervised_datasets,
    build_unlabeled_datasets,
    build_dataloaders,
)
from src.train import run_training
from src.evaluate import evaluate_checkpoint
from src.visualization import show_dataset_examples, show_predictions

# ── Debug fingerprint helpers ─────────────────────────────────
import json, os, platform, subprocess, time, importlib.metadata

def _fp_get_version(pkg):
    try: return importlib.metadata.version(pkg)
    except Exception: return None

def _fp_git_hash_from_src(src_train_file):
    # Derive repo root from src/train.py: {repo_root}/src/train.py
    repo_root = os.path.dirname(os.path.dirname(os.path.abspath(src_train_file)))
    try:
        return subprocess.check_output(
            ["git", "log", "--oneline", "-1"], cwd=repo_root, stderr=subprocess.DEVNULL
        ).decode().strip()
    except Exception: return None

def _fp_save(pre, post, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump({"pre_run": pre, "post_run": post}, f, indent=2, default=str)

def _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                    unlabeled_ds=None, temporal_unlab_ds=None):
    import torch, sys
    from src.models import create_model

    # 1. Environment
    try: gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "n/a"
    except: gpu_name = "n/a"
    env = {
        "python":      sys.version,
        "torch":       torch.__version__,
        "torchvision": _fp_get_version("torchvision"),
        "cuda":        torch.version.cuda,
        "cudnn":       str(torch.backends.cudnn.version()) if torch.cuda.is_available() else "n/a",
        "segmentation_models_pytorch": _fp_get_version("segmentation-models-pytorch"),
        "albumentations": _fp_get_version("albumentations"),
        "platform":    platform.platform(),
        "gpu_name":    gpu_name,
    }

    # 2. Code provenance — git hash derived from actual runtime src path
    import src.train, src.datasets, src.defaults, src.evaluate
    provenance = {
        "src_train":    src.train.__file__,
        "src_datasets": src.datasets.__file__,
        "src_defaults": src.defaults.__file__,
        "src_evaluate": src.evaluate.__file__,
        "git_hash":     _fp_git_hash_from_src(src.train.__file__),
    }

    # 3. Effective config
    cfg_keys = [
        "seed", "arch", "backbone", "n_classes",
        "image_preproc", "mask_smoothing", "target_size", "use_pad", "imagenet_norm",
        "batch_size", "num_workers", "drop_last", "num_augmented",
        "lr", "weight_decay", "epochs", "warmup_epochs", "patience_es", "eval_threshold",
        "use_semi", "use_temp_consistency",
        "lambda_u", "tau", "ema_decay", "semi_start_epoch", "semi_warmup_epochs", "lambda_t",
        "unlabeled_subdir", "exp_dir",
    ]
    eff_cfg = {k: cfg.get(k) for k in cfg_keys}
    unlab_loader = loaders.get("unlabeled_loader")
    eff_cfg["batch_size_unlab"] = unlab_loader.batch_size if unlab_loader is not None else None

    # 4. Dataset / loader facts
    ds_facts = {
        "len_train_ds":          len(train_ds),
        "len_val_ds":            len(val_ds),
        "len_test_ds":           len(test_ds),
        "len_unlabeled_ds":      len(unlabeled_ds) if unlabeled_ds is not None else None,
        "len_temporal_unlab_ds": len(temporal_unlab_ds) if temporal_unlab_ds is not None else None,
        "train_loader_batch_size":      loaders["train_loader"].batch_size,
        "train_loader_num_workers":     loaders["train_loader"].num_workers,
        "train_loader_drop_last":       loaders["train_loader"].drop_last,
        "unlabeled_loader_batch_size":  unlab_loader.batch_size if unlab_loader else None,
        "unlabeled_loader_num_workers": unlab_loader.num_workers if unlab_loader else None,
        "unlabeled_loader_drop_last":   unlab_loader.drop_last if unlab_loader else None,
    }

    # 5. Sample identifiers — reads .files attribute, no IO beyond what dataset already did
    try: sup_ids = train_ds.files[:5]
    except Exception as e: sup_ids = f"unavailable: {e}"
    try: unl_ids = unlabeled_ds.files[:5] if unlabeled_ds is not None else None
    except Exception as e: unl_ids = f"unavailable: {e}"
    sample_ids = {"first5_train": sup_ids, "first5_unlabeled": unl_ids}

    # 6. Batch tensor shapes — analytical, no DataLoader consumed, no RNG touched
    try:
        H, W = cfg["target_size"]
        C = 3  # IMREAD_COLOR: grayscale PNGs expand to 3 identical channels
        eff_bs = cfg["batch_size"] * (1 + cfg.get("num_augmented", 0))  # flatten_collate
        bs_u = max(1, cfg["batch_size"] // 4)  # mirrors datasets.py build_dataloaders
        batch_shapes = {
            "xb":   [eff_bs, C, H, W],
            "yb":   [eff_bs, 1, H, W],
            "xw_u": [bs_u, C, H, W] if unlab_loader is not None else None,
            "xs_u": [bs_u, C, H, W] if unlab_loader is not None else None,
            "note": "analytically derived from cfg — no DataLoader consumed",
        }
    except Exception as e:
        batch_shapes = {"error": str(e)}

    # 7. Model fingerprint — RNG save/restore so training is unaffected.
    # Belt-and-suspenders: run_training() also calls seed_everything(seed) first.
    try:
        import random as _random, numpy as _np
        _rng = {
            "py":   _random.getstate(),
            "np":   _np.random.get_state(),
            "th":   torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        }
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        model_fp = {
            "total_params":          sum(p.numel() for p in _m.parameters()),
            "trainable_params":      sum(p.numel() for p in _m.parameters() if p.requires_grad),
            "first_state_dict_keys": list(_m.state_dict().keys())[:8],
        }
        del _m
        _random.setstate(_rng["py"])
        _np.random.set_state(_rng["np"])
        torch.set_rng_state(_rng["th"])
        if _rng["cuda"] is not None:
            torch.cuda.set_rng_state_all(_rng["cuda"])
    except Exception as e:
        model_fp = {"error": str(e)}

    return {
        "timestamp_utc":     time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "environment":       env,
        "provenance":        provenance,
        "effective_cfg":     eff_cfg,
        "dataset_facts":     ds_facts,
        "sample_ids":        sample_ids,
        "batch_shapes":      batch_shapes,
        "model_fingerprint": model_fp,
    }


def _fp_collect_post(artifacts, results):
    history = artifacts.get("history") or []
    best_row = max(history, key=lambda r: r.get("val_iou_global", 0.0)) if history else None
    vm = (results or {}).get("val_metrics", {})
    tm = (results or {}).get("test_metrics", {})
    return {
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "best_path":     artifacts.get("best_path"),
        "best_epoch_info": {
            "epoch":          best_row.get("epoch") if best_row else None,
            "val_iou_global": best_row.get("val_iou_global") if best_row else None,
            "val_loss":       best_row.get("val_loss") if best_row else None,
        },
        "val_metrics": {
            "f1_global":       vm.get("global_f1"),
            "iou_global":      vm.get("global_iou"),
            "f1_sample_mean":  vm.get("sample_mean_f1"),
            "iou_sample_mean": vm.get("sample_mean_iou"),
        },
        "test_metrics": {
            "f1_global":       tm.get("global_f1"),
            "iou_global":      tm.get("global_iou"),
            "f1_sample_mean":  tm.get("sample_mean_f1"),
            "iou_sample_mean": tm.get("sample_mean_iou"),
        },
        "finished_successfully": True,
        "exception": None,
    }
# ──────────────────────────────────────────────────────────────

print("Imports OK — select one experiment cell below and run it.")

## Experiment: supervised_inca

Supervised baseline on INCA dataset (3 seeds).


In [ ]:
# === RUN 1/81: supervised_inca/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "supervised_inca"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5
cfg["run_ruler_eval"] = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "supervised_inca_seed_0_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=None,
        temporal_unlab_ds=None,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=None, temporal_unlab_ds=None)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 2/81: supervised_inca/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "supervised_inca"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5
cfg["run_ruler_eval"] = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "supervised_inca_seed_1_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=None,
        temporal_unlab_ds=None,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=None, temporal_unlab_ds=None)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 3/81: supervised_inca/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "supervised_inca"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only
cfg["seed"]                 = _SEED
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5
cfg["run_ruler_eval"] = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "supervised_inca_seed_2_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf = get_supervised_train_augmentation(cfg)
    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=None,
        temporal_unlab_ds=None,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=None, temporal_unlab_ds=None)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Experiment: semi_inca_r3

Pseudo-label SSL, temporal pool r=3 on INCA.


In [ ]:
# === RUN 4/81: semi_inca_r3/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_r3"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r3"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_r3_seed_0_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 5/81: semi_inca_r3/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_r3"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r3"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_r3_seed_1_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 6/81: semi_inca_r3/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_r3"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r3"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_r3_seed_2_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Experiment: semi_inca_r5

Pseudo-label SSL, temporal pool r=5 on INCA.


In [ ]:
# === RUN 7/81: semi_inca_r5/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_r5"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r5"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_r5_seed_0_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 8/81: semi_inca_r5/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_r5"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r5"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_r5_seed_1_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 9/81: semi_inca_r5/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_r5"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r5"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_r5_seed_2_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Experiment: semi_inca_r7

Pseudo-label SSL, temporal pool r=7 on INCA.


In [ ]:
# === RUN 10/81: semi_inca_r7/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_r7"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r7"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_r7_seed_0_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 11/81: semi_inca_r7/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_r7"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r7"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_r7_seed_1_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 12/81: semi_inca_r7/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_r7"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r7"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_r7_seed_2_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Experiment: semi_inca_r10

Pseudo-label SSL, temporal pool r=10 on INCA.


In [ ]:
# === RUN 13/81: semi_inca_r10/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_r10"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r10"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_r10_seed_0_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 14/81: semi_inca_r10/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_r10"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r10"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_r10_seed_1_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 15/81: semi_inca_r10/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_r10"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r10"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_r10_seed_2_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Experiment: semi_inca_r15

Pseudo-label SSL, temporal pool r=15 on INCA.


In [ ]:
# === RUN 16/81: semi_inca_r15/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_r15"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r15"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_r15_seed_0_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 17/81: semi_inca_r15/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_r15"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r15"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_r15_seed_1_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 18/81: semi_inca_r15/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_r15"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r15"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_r15_seed_2_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Experiment: semi_inca_r20

Pseudo-label SSL, temporal pool r=20 on INCA.


In [ ]:
# === RUN 19/81: semi_inca_r20/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_r20"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r20"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_r20_seed_0_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 20/81: semi_inca_r20/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_r20"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r20"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_r20_seed_1_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 21/81: semi_inca_r20/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_r20"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r20"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_r20_seed_2_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Experiment: semi_inca_std_matched_r3

Pseudo-label SSL, random matched pool r=3 on INCA.


In [ ]:
# === RUN 22/81: semi_inca_std_matched_r3/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_std_matched_r3"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r3"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_std_matched_r3_seed_0_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 23/81: semi_inca_std_matched_r3/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_std_matched_r3"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r3"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_std_matched_r3_seed_1_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 24/81: semi_inca_std_matched_r3/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_std_matched_r3"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r3"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_std_matched_r3_seed_2_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Experiment: semi_inca_std_matched_r5

Pseudo-label SSL, random matched pool r=5 on INCA.


In [ ]:
# === RUN 25/81: semi_inca_std_matched_r5/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_std_matched_r5"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r5"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_std_matched_r5_seed_0_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 26/81: semi_inca_std_matched_r5/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_std_matched_r5"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r5"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_std_matched_r5_seed_1_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 27/81: semi_inca_std_matched_r5/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_std_matched_r5"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r5"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_std_matched_r5_seed_2_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Experiment: semi_inca_std_matched_r7

Pseudo-label SSL, random matched pool r=7 on INCA.


In [ ]:
# === RUN 28/81: semi_inca_std_matched_r7/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_std_matched_r7"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r7"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_std_matched_r7_seed_0_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 29/81: semi_inca_std_matched_r7/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_std_matched_r7"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r7"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_std_matched_r7_seed_1_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 30/81: semi_inca_std_matched_r7/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_std_matched_r7"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r7"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_std_matched_r7_seed_2_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Experiment: semi_inca_std_matched_r10

Pseudo-label SSL, random matched pool r=10 on INCA.


In [ ]:
# === RUN 31/81: semi_inca_std_matched_r10/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_std_matched_r10"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r10"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_std_matched_r10_seed_0_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 32/81: semi_inca_std_matched_r10/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_std_matched_r10"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r10"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_std_matched_r10_seed_1_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 33/81: semi_inca_std_matched_r10/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_std_matched_r10"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r10"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_std_matched_r10_seed_2_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Experiment: semi_inca_std_matched_r15

Pseudo-label SSL, random matched pool r=15 on INCA.


In [ ]:
# === RUN 34/81: semi_inca_std_matched_r15/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_std_matched_r15"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r15"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_std_matched_r15_seed_0_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 35/81: semi_inca_std_matched_r15/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_std_matched_r15"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r15"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_std_matched_r15_seed_1_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 36/81: semi_inca_std_matched_r15/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_std_matched_r15"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r15"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_std_matched_r15_seed_2_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Experiment: semi_inca_std_matched_r20

Pseudo-label SSL, random matched pool r=20 on INCA.


In [ ]:
# === RUN 37/81: semi_inca_std_matched_r20/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_std_matched_r20"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r20"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_std_matched_r20_seed_0_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 38/81: semi_inca_std_matched_r20/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_std_matched_r20"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r20"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_std_matched_r20_seed_1_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 39/81: semi_inca_std_matched_r20/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_std_matched_r20"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r20"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_std_matched_r20_seed_2_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Experiment: semi_inca_all_lateral

Pseudo-label SSL, full lateral pool on INCA.


In [ ]:
# === RUN 40/81: semi_inca_all_lateral/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_all_lateral"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_all"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_all_lateral_seed_0_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 41/81: semi_inca_all_lateral/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_all_lateral"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_all"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_all_lateral_seed_1_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 42/81: semi_inca_all_lateral/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "semi_inca_all_lateral"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "pseudo_label"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_all"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "semi_inca_all_lateral_seed_2_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Experiment: mean_teacher_inca_r3

Mean Teacher SSL, temporal pool r=3 on INCA.


In [ ]:
# === RUN 43/81: mean_teacher_inca_r3/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_r3"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r3"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_r3_seed_0_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 44/81: mean_teacher_inca_r3/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_r3"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r3"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_r3_seed_1_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 45/81: mean_teacher_inca_r3/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_r3"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r3"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_r3_seed_2_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Experiment: mean_teacher_inca_r5

Mean Teacher SSL, temporal pool r=5 on INCA.


In [ ]:
# === RUN 46/81: mean_teacher_inca_r5/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_r5"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r5"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_r5_seed_0_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 47/81: mean_teacher_inca_r5/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_r5"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r5"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_r5_seed_1_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 48/81: mean_teacher_inca_r5/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_r5"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r5"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_r5_seed_2_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Experiment: mean_teacher_inca_r7

Mean Teacher SSL, temporal pool r=7 on INCA.


In [ ]:
# === RUN 49/81: mean_teacher_inca_r7/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_r7"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r7"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_r7_seed_0_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 50/81: mean_teacher_inca_r7/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_r7"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r7"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_r7_seed_1_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 51/81: mean_teacher_inca_r7/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_r7"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r7"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_r7_seed_2_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Experiment: mean_teacher_inca_r10

Mean Teacher SSL, temporal pool r=10 on INCA.


In [ ]:
# === RUN 52/81: mean_teacher_inca_r10/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_r10"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r10"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_r10_seed_0_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 53/81: mean_teacher_inca_r10/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_r10"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r10"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_r10_seed_1_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 54/81: mean_teacher_inca_r10/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_r10"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r10"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_r10_seed_2_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Experiment: mean_teacher_inca_r15

Mean Teacher SSL, temporal pool r=15 on INCA.


In [ ]:
# === RUN 55/81: mean_teacher_inca_r15/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_r15"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r15"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_r15_seed_0_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 56/81: mean_teacher_inca_r15/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_r15"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r15"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_r15_seed_1_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 57/81: mean_teacher_inca_r15/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_r15"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r15"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_r15_seed_2_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Experiment: mean_teacher_inca_r20

Mean Teacher SSL, temporal pool r=20 on INCA.


In [ ]:
# === RUN 58/81: mean_teacher_inca_r20/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_r20"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r20"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_r20_seed_0_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 59/81: mean_teacher_inca_r20/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_r20"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r20"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_r20_seed_1_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 60/81: mean_teacher_inca_r20/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_r20"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_r20"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_r20_seed_2_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Experiment: mean_teacher_inca_std_matched_r3

Mean Teacher SSL, random matched pool r=3 on INCA.


In [ ]:
# === RUN 61/81: mean_teacher_inca_std_matched_r3/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_std_matched_r3"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r3"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_std_matched_r3_seed_0_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 62/81: mean_teacher_inca_std_matched_r3/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_std_matched_r3"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r3"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_std_matched_r3_seed_1_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 63/81: mean_teacher_inca_std_matched_r3/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_std_matched_r3"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r3"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_std_matched_r3_seed_2_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Experiment: mean_teacher_inca_std_matched_r5

Mean Teacher SSL, random matched pool r=5 on INCA.


In [ ]:
# === RUN 64/81: mean_teacher_inca_std_matched_r5/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_std_matched_r5"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r5"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_std_matched_r5_seed_0_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 65/81: mean_teacher_inca_std_matched_r5/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_std_matched_r5"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r5"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_std_matched_r5_seed_1_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 66/81: mean_teacher_inca_std_matched_r5/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_std_matched_r5"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r5"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_std_matched_r5_seed_2_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Experiment: mean_teacher_inca_std_matched_r7

Mean Teacher SSL, random matched pool r=7 on INCA.


In [ ]:
# === RUN 67/81: mean_teacher_inca_std_matched_r7/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_std_matched_r7"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r7"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_std_matched_r7_seed_0_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 68/81: mean_teacher_inca_std_matched_r7/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_std_matched_r7"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r7"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_std_matched_r7_seed_1_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 69/81: mean_teacher_inca_std_matched_r7/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_std_matched_r7"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r7"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_std_matched_r7_seed_2_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Experiment: mean_teacher_inca_std_matched_r10

Mean Teacher SSL, random matched pool r=10 on INCA.


In [ ]:
# === RUN 70/81: mean_teacher_inca_std_matched_r10/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_std_matched_r10"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r10"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_std_matched_r10_seed_0_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 71/81: mean_teacher_inca_std_matched_r10/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_std_matched_r10"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r10"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_std_matched_r10_seed_1_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 72/81: mean_teacher_inca_std_matched_r10/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_std_matched_r10"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r10"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_std_matched_r10_seed_2_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Experiment: mean_teacher_inca_std_matched_r15

Mean Teacher SSL, random matched pool r=15 on INCA.


In [ ]:
# === RUN 73/81: mean_teacher_inca_std_matched_r15/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_std_matched_r15"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r15"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_std_matched_r15_seed_0_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 74/81: mean_teacher_inca_std_matched_r15/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_std_matched_r15"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r15"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_std_matched_r15_seed_1_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 75/81: mean_teacher_inca_std_matched_r15/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_std_matched_r15"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r15"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_std_matched_r15_seed_2_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Experiment: mean_teacher_inca_std_matched_r20

Mean Teacher SSL, random matched pool r=20 on INCA.


In [ ]:
# === RUN 76/81: mean_teacher_inca_std_matched_r20/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_std_matched_r20"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r20"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_std_matched_r20_seed_0_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 77/81: mean_teacher_inca_std_matched_r20/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_std_matched_r20"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r20"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_std_matched_r20_seed_1_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 78/81: mean_teacher_inca_std_matched_r20/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_std_matched_r20"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_random_r20"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_std_matched_r20_seed_2_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


## Experiment: mean_teacher_inca_all_lateral

Mean Teacher SSL, full lateral pool on INCA.


In [ ]:
# === RUN 79/81: mean_teacher_inca_all_lateral/seed_0 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_all_lateral"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_all"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_all_lateral_seed_0_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 80/81: mean_teacher_inca_all_lateral/seed_1 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_all_lateral"
_SEED     = 1
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_all"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_all_lateral_seed_1_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise


In [ ]:
# === RUN 81/81: mean_teacher_inca_all_lateral/seed_2 ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3/data/inca_dataset"
cfg["rotulos_dir"] = ""
_EXP_NAME = "mean_teacher_inca_all_lateral"
_SEED     = 2
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_inca_v2/{_EXP_NAME}/seed_{_SEED}"
cfg["dataset"] = "inca"
cfg["ssl_method"] = "mean_teacher"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 10
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeled_all"
cfg["run_ruler_eval"]   = True

# Skip if already completed
_report_path = os.path.join(cfg["exp_dir"], "mean_teacher_inca_all_lateral_seed_2_run_report.json")
if os.path.isfile(_report_path):
    print(f"SKIP: {_EXP_NAME}/seed_{_SEED} — report exists: {_report_path}")
else:
    print(summarize_config(cfg))

    train_tf  = get_supervised_train_augmentation(cfg)
    weak_tf   = get_weak_augmentation(cfg)
    strong_tf = get_strong_augmentation(cfg)

    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
        cfg, weak_tf=weak_tf, strong_tf=strong_tf
    )

    loaders = build_dataloaders(
        cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        unlabeled_ds=unlabeled_ds,
        temporal_unlab_ds=temporal_unlab_ds,
    )

    _fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
    _fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)
    _fp_save(_fp_pre, None, _fp_path)
    print(f"[fingerprint] pre_run saved → {_fp_path}")

    try:
        artifacts = run_training(cfg, loaders)
        results = evaluate_checkpoint(
            cfg,
            artifacts['model'],
            loaders,
            artifacts['best_path'],
            artifacts['history'],
        )
        _fp_post = _fp_collect_post(artifacts, results)
        _fp_save(_fp_pre, _fp_post, _fp_path)
        print(f"[fingerprint] post_run saved → {_fp_path}")
        print(results)
    except Exception as _fp_exc:
        import traceback
        _fp_save(_fp_pre, {
            "finished_successfully": False,
            "exception": traceback.format_exc(),
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, _fp_path)
        print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
        raise
